# OTP Analysis: Matchups and Opponent Specialization

This notebook enriches the existing OTP dataset. It does not rebuild the seed cohort, rerun seed histories, request timelines, or change the raw source dataset. Persistent opponent history state and the shared raw-match cache make it safe to resume.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

import otp_match_pipeline as otp

load_dotenv()
RIOT_API_KEY = os.getenv('RIOT_API_KEY')
HEADERS = {'X-Riot-Token': RIOT_API_KEY} if RIOT_API_KEY else None

TARGET_PATCH = "16.19"
MATCHES_PER_OPPONENT = 100
MAX_NEW_OPPONENT_DOWNLOADS_PER_RUN = 2000

# Existing indexed opponents are always skipped on later runs.
RUN_OPPONENT_COLLECTION = False

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RAW_MATCH_DIR = PROJECT_DIR / 'data' / 'otp_raw_matches'

## Load existing source tables

Parquet is preferred for both source tables. The OTP table is never rebuilt from the Riot API here.

In [2]:
otp_matches = otp.load_preferred_table(PROCESSED_DIR, 'otp_matches_16_19')
expected_winrates = otp.load_preferred_table(PROCESSED_DIR, 'expected_winrates_16_19')

if otp_matches['patch'].astype(str).ne(TARGET_PATCH).any():
    raise ValueError(f'otp_matches contains a patch other than {TARGET_PATCH}.')

print(f'Loaded OTP rows: {len(otp_matches):,}')
print(f'Loaded expected-winrate rows: {len(expected_winrates):,}')

Loaded OTP rows: 129
Loaded expected-winrate rows: 121


## Merge successful LoLalytics matchup data

Only rows whose scraper status is `ok` are used. Missing combinations are saved for a future resumable scraper run; this notebook never scrapes them.

In [3]:
expected_lookup = otp.prepare_expected_matchups(expected_winrates, TARGET_PATCH)
otp_with_expected = otp.merge_expected_matchups(otp_matches, expected_lookup)
missing_matchups = otp.build_missing_matchups(otp_matches, expected_lookup)
otp.save_missing_matchups(missing_matchups, PROCESSED_DIR, TARGET_PATCH)

print(f'Total OTP rows: {len(otp_with_expected):,}')
print(f"Rows with expected winrate: {otp_with_expected['expected_winrate'].notna().sum():,}")
print(f"Rows missing expected winrate: {otp_with_expected['expected_winrate'].isna().sum():,}")
print(f'Missing matchup combinations saved: {len(missing_matchups):,}')

Total OTP rows: 129
Rows with expected winrate: 127
Rows missing expected winrate: 2
Missing matchup combinations saved: 2


## Resumable opponent histories

Known seed-player histories are copied into the opponent index before API calls. Only opponents with no persisted history rows are requested; every successful response is saved immediately.

In [4]:
opponent_puuids = otp_matches['opponent_puuid'].dropna().drop_duplicates().tolist()
player_match_index = otp.load_preferred_table(PROCESSED_DIR, 'player_match_index')
opponent_match_index = otp.load_opponent_match_index(PROCESSED_DIR)

opponents_already_indexed = set(opponent_match_index['opponent_puuid']).intersection(opponent_puuids)
opponent_match_index = otp.add_seed_histories_to_opponent_index(
    opponent_match_index, opponent_puuids, player_match_index
)
opponent_match_index = otp.save_opponent_match_index(opponent_match_index, PROCESSED_DIR)

opponents_needing_history = [
    puuid for puuid in opponent_puuids
    if puuid not in set(opponent_match_index['opponent_puuid'])
]
opponent_history_requests = 0

if RUN_OPPONENT_COLLECTION and opponents_needing_history:
    if HEADERS is None:
        raise RuntimeError('Set RIOT_API_KEY in .env before enabling opponent collection.')
    opponent_match_index, _, opponent_history_requests = otp.append_unindexed_opponent_histories(
        opponent_match_index,
        opponent_puuids,
        HEADERS,
        MATCHES_PER_OPPONENT,
        PROCESSED_DIR,
    )
else:
    print('No new opponent history calls were needed.')

opponent_match_index = otp.save_opponent_match_index(opponent_match_index, PROCESSED_DIR)
print(f'Unique opponents: {len(opponent_puuids):,}')
print(f'Opponents already indexed: {len(opponents_already_indexed):,}')
print(f'Seed-opponent histories reused: {len(opponent_puuids) - len(opponents_needing_history) - len(opponents_already_indexed):,}')
print(f'New opponent histories requested: {len(opponents_needing_history):,}')
print(f'Successful opponent history requests: {opponent_history_requests:,}')

No new opponent history calls were needed.
Unique opponents: 121
Opponents already indexed: 121
Seed-opponent histories reused: 0
New opponent histories requested: 0
Successful opponent history requests: 0


## Reuse the raw match cache

The existing downloader is reused with the opponent index. Files already in `data/otp_raw_matches/` are skipped, and the cap counts only new Match-V5 payloads.

In [5]:
cached_match_ids = {path.stem for path in RAW_MATCH_DIR.glob('*.json')}
opponent_match_ids = set(opponent_match_index['match_id'].dropna())
cached_matches_reused = len(cached_match_ids)
opponent_matches_still_missing = len(opponent_match_ids - cached_match_ids)
new_opponent_downloads_this_run = 0

if RUN_OPPONENT_COLLECTION and not opponent_match_index.empty:
    opponent_download_index = opponent_match_index.assign(seed_tier='opponent')
    cached_matches_reused, opponent_matches_still_missing, new_opponent_downloads_this_run = (
        otp.download_missing_matches(
            opponent_download_index,
            RAW_MATCH_DIR,
            HEADERS,
            MAX_NEW_OPPONENT_DOWNLOADS_PER_RUN,
        )
    )

print(f'Cached matches reused: {cached_matches_reused:,}')
print(f'Opponent matches still missing: {opponent_matches_still_missing:,}')
print(f'New match downloads this run: {new_opponent_downloads_this_run:,}')

Cached matches reused: 2,083
Opponent matches still missing: 10,902
New match downloads this run: 0


## Build and save the final analysis dataframe

Opponent specialization scans all usable cached Ranked Solo/Duo patch-16.19 games for each current opponent, regardless of whether lane matching succeeded. Rows without cached opponent coverage are retained.

In [6]:
opponent_specialization, unreadable_cached_files = otp.build_opponent_specialization(
    opponent_puuids, RAW_MATCH_DIR, TARGET_PATCH
)
otp.save_opponent_specialization(opponent_specialization, PROCESSED_DIR, TARGET_PATCH)

analysis_df = otp.finalize_analysis_dataframe(
    otp_matches, expected_lookup, opponent_specialization
)
otp.save_analysis_dataframe(analysis_df, PROCESSED_DIR, TARGET_PATCH)

print(f'Unreadable cached files skipped during opponent specialization: {unreadable_cached_files:,}')

Unreadable cached files skipped during opponent specialization: 0


## Summary and validation

In [7]:
tier_rows = analysis_df['seed_tier'].value_counts()

print(f'Final analysis rows: {len(analysis_df):,}')
print(f"Unique matches: {analysis_df['match_id'].nunique():,}")
print(f"Unique seed players: {analysis_df['puuid'].nunique():,}")
print(f"Unique opponents: {analysis_df['opponent_puuid'].nunique():,}")
print(f"Rows with champion_share: {analysis_df['champion_share'].notna().sum():,}")
print(f"Rows with opponent_champion_share: {analysis_df['opponent_champion_share'].notna().sum():,}")
print(f"Rows with expected_winrate: {analysis_df['expected_winrate'].notna().sum():,}")
print(f"Rows missing opponent_champion_share: {analysis_df['opponent_champion_share'].isna().sum():,}")
print(f"Rows missing expected_winrate: {analysis_df['expected_winrate'].isna().sum():,}")
print(f"Master rows: {tier_rows.get('master', 0):,}")
print(f"Grandmaster rows: {tier_rows.get('grandmaster', 0):,}")
print(f"Challenger rows: {tier_rows.get('challenger', 0):,}")

display(analysis_df.head(20))
display(analysis_df.shape)
display(analysis_df.dtypes)
display(analysis_df[[
    'champion',
    'opponent_champion',
    'champion_share',
    'opponent_champion_share',
    'expected_winrate',
    'matchup_games',
    'win',
]].head(20))

Final analysis rows: 129
Unique matches: 125
Unique seed players: 14
Unique opponents: 121
Rows with champion_share: 129
Rows with opponent_champion_share: 129
Rows with expected_winrate: 127
Rows missing opponent_champion_share: 0
Rows missing expected_winrate: 2
Master rows: 129
Grandmaster rows: 0
Challenger rows: 0


,match_id,puuid,seed_tier,patch,side,lane,champion,opponent_champion,opponent_puuid,win,...,player_games,champion_share,matchup_rank,expected_winrate,matchup_games,opponent_champion_games,opponent_player_games,opponent_champion_share,expected_winrate_prob,specialization_diff
0,NA1_5647510258,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vladimir,5dwOvvWd4lI2z8arXUql3b7gyhv-fCFfMSU5xbA9fJ_9rD...,False,...,10,0.900000,emerald,52.31,65.0,1,1,1.000000,0.5231,-0.100000
1,NA1_5647497734,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Anivia,J3ZfdIi9Q8h5xKzPAlYvOhAOM1OpE85aM1Rv6M-leAsHK3...,False,...,10,0.900000,emerald,38.89,36.0,1,1,1.000000,0.3889,-0.100000
2,NA1_5647517399,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,jungle,Nunu,FiddleSticks,qIRl10tZIWyJcyrGwScd1YJ9vXekf_ubDWt7bO054EvF9W...,False,...,19,0.894737,emerald,51.79,56.0,1,2,0.500000,0.5179,0.394737
3,NA1_5647521561,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Vex,Ts1tliuj-uuhtj4zLUkYRkjRSuVoFR5zAg5sAV2dfLmGWT...,False,...,10,0.900000,emerald,53.49,43.0,2,3,0.666667,0.5349,0.233333
4,NA1_5647532032,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,red,mid,Lucian,Brand,rj980nimbBPg1vHHy3vjP-ocQrXbx8YTDtJf-yMIEWL-SB...,True,...,19,0.052632,emerald,52.38,21.0,1,2,0.500000,0.5238,-0.447368
5,NA1_5647527197,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,mid,Taliyah,Locke,x_vzZsz_xa7Mn5jrtm5v_QWb6SDAH64-4xcq1Je7NDFSsC...,False,...,10,0.900000,emerald,55.84,77.0,2,2,1.000000,0.5584,-0.100000
6,NA1_5647533561,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Zed,2QXWlza82YF20hVJRFcsbWsCMSbSb6bhcdHTnLBu-J5R3c...,False,...,19,0.894737,emerald,53.45,232.0,1,2,0.500000,0.5345,0.394737
7,NA1_5647531679,ToJzMBS3nrB_0-q0kX5OyoTSxLL7ZgKHjpkO3fQiyMhEvB...,master,16.19,red,adc,Mel,Swain,FDMy3Wi1Y_xxJesQc0jk1TikcROYbqotKHQLyoXNPm8GpE...,False,...,10,0.100000,emerald,52.83,53.0,1,1,1.000000,0.5283,-0.900000
8,NA1_5647554713,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Corki,A1BlMq00EDwNzf-5SmzXxVP7RydiGEHcdZodbPIFWr4fWG...,True,...,19,0.894737,emerald,51.28,39.0,1,1,1.000000,0.5128,-0.105263
9,NA1_5647573128,0wanq1XDJV3BdiIPRHsEFxngU82mqcS7DJWWPWBHJgLscM...,master,16.19,blue,jungle,Nunu,Hecarim,2-GsGYJnr1_9Yqh0_Am0TEvjn-spPerzWt6mJAvMXsIqlg...,True,...,19,0.894737,emerald,60.87,138.0,1,3,0.333333,0.6087,0.561404


(129, 23)

match_id                    object
puuid                       object
seed_tier                   object
patch                       object
side                        object
lane                        object
champion                    object
opponent_champion           object
opponent_puuid              object
win                           bool
game_duration                int64
game_start_timestamp         int64
champion_games               int64
player_games                 int64
champion_share             float64
matchup_rank                object
expected_winrate           float64
matchup_games              float64
opponent_champion_games      int64
opponent_player_games        int64
opponent_champion_share    float64
expected_winrate_prob      float64
specialization_diff        float64
dtype: object

,champion,opponent_champion,champion_share,opponent_champion_share,expected_winrate,matchup_games,win
0,Taliyah,Vladimir,0.900000,1.000000,52.31,65.0,False
1,Taliyah,Anivia,0.900000,1.000000,38.89,36.0,False
2,Nunu,FiddleSticks,0.894737,0.500000,51.79,56.0,False
3,Taliyah,Vex,0.900000,0.666667,53.49,43.0,False
4,Lucian,Brand,0.052632,0.500000,52.38,21.0,True
5,Taliyah,Locke,0.900000,1.000000,55.84,77.0,False
6,Nunu,Zed,0.894737,0.500000,53.45,232.0,False
7,Mel,Swain,0.100000,1.000000,52.83,53.0,False
8,Nunu,Corki,0.894737,1.000000,51.28,39.0,True
9,Nunu,Hecarim,0.894737,0.333333,60.87,138.0,True
